In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import pandas as pd

path = "/kaggle/input/competitions/playground-series-s6e7"

train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

(690088, 15)
(295753, 14)


,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [2]:
X = train.drop(columns=["health_condition", "id"])
y = train["health_condition"]

X_test = test.drop(columns=["id"])

print(X.shape)
print(y.shape)
print(X_test.shape)

(690088, 13)
(690088,)
(295753, 13)


In [5]:
import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

In [6]:
import numpy as np

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)
print("Encoded labels:", np.unique(y_encoded))

Classes: ['at-risk' 'fit' 'unhealthy']
Encoded labels: [0 1 2]


In [7]:
combined = pd.concat([X, X_test], axis=0)

combined = pd.get_dummies(

    combined,

    columns=combined.select_dtypes(include=["object", "category"]).columns,

    dummy_na=True

)

X_encoded = combined.iloc[:len(X)].copy()

X_test_encoded = combined.iloc[len(X):].copy()

print(X_encoded.shape)

print(X_test_encoded.shape)

(690088, 31)
(295753, 31)


## 5. Stratified Cross-Validation Setup

We use 5-fold stratified cross-validation to preserve the class distribution in every fold.  
We also create containers for out-of-fold predictions, averaged test probabilities, and fold scores.

In [10]:
# Create 5 stratified folds
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store validation predictions for every training row
oof_predictions = np.zeros(
    len(X_encoded),
    dtype=int
)

# Store averaged class probabilities for the test set
test_probabilities = np.zeros(
    (len(X_test_encoded), len(label_encoder.classes_))
)

# Store balanced accuracy for each fold
fold_scores = []

## 6. Train XGBoost Models

We train one XGBoost model for each fold, evaluate it on the validation fold, and average the test-set probabilities across all five models.

In [11]:
for fold, (train_index, valid_index) in enumerate(
    skf.split(X_encoded, y_encoded),
    start=1
):
    print(f"\nFold {fold}")

    # Split the current fold into training and validation sets
    X_train = X_encoded.iloc[train_index]
    X_valid = X_encoded.iloc[valid_index]

    y_train = y_encoded[train_index]
    y_valid = y_encoded[valid_index]

    # Create the XGBoost multiclass model
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    # Train the model
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=100
    )

    # Predict validation labels
    valid_predictions = model.predict(X_valid)

    # Calculate balanced accuracy for this fold
    fold_score = balanced_accuracy_score(
        y_valid,
        valid_predictions
    )

    fold_scores.append(fold_score)
    oof_predictions[valid_index] = valid_predictions

    # Average test probabilities across all folds
    test_probabilities += (
        model.predict_proba(X_test_encoded) / skf.n_splits
    )

    print(f"Balanced accuracy: {fold_score:.6f}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.

Fold 1
[0]	validation_0-mlogloss:0.45308
[100]	validation_0-mlogloss:0.09244
[200]	validation_0-mlogloss:0.08934
[300]	validation_0-mlogloss:0.08891
[400]	validation_0-mlogloss:0.08881
[500]	validation_0-mlogloss:0.08879
[600]	validation_0-mlogloss:0.08885
[700]	validation_0-mlogloss:0.08895
[800]	validation_0-mlogloss:0.08910
[900]	validation_0-mlogloss:0.08923
[999]	validation_0-mlogloss:0.08937
Balanced accuracy: 0.879219

Fold 2
[0]	validation_0-mlogloss:0.45293
[100]	validation_0-mlogloss:0.09183
[200]	validation_0-mlogloss:0.08855
[300]	validation_0-mlogloss:0.08803
[400]	validation_0-mlogloss:0.08790
[500]	validation_0-mlogloss:0.08784
[600]	validation_0-mlogloss:0.08789
[700]	validation_0-mlogloss:0.08791
[800]	validation_0-mlogloss:0.08800
[900]	validation_0-mlogloss:0.08815
[999]	validation_0-mlogloss:0.08825
Balanced accur

## 7. Evaluate Cross-Validation Results

We calculate the mean fold score, standard deviation, and overall out-of-fold balanced accuracy.

In [13]:
print("Fold scores:", fold_scores)
print("Mean CV balanced accuracy:", np.mean(fold_scores))
print("CV standard deviation:", np.std(fold_scores))

overall_oof_score = balanced_accuracy_score(
    y_encoded,
    oof_predictions
)

print("Overall OOF balanced accuracy:", overall_oof_score)

Fold scores: [np.float64(0.8792185619111584), np.float64(0.8828244608605725), np.float64(0.8843667580146898), np.float64(0.8794563048895614), np.float64(0.87937185464236)]
Mean CV balanced accuracy: 0.8810475880636683
CV standard deviation: 0.0021382126838176137
Overall OOF balanced accuracy: 0.8810476407427043


## 8. Create Submission

We average the test probabilities from all folds, convert them back to class labels, and save the final Kaggle submission file.

In [14]:
# Convert averaged probabilities into encoded class predictions
test_predictions_encoded = np.argmax(
    test_probabilities,
    axis=1
)

# Convert encoded predictions back to original labels
test_predictions = label_encoder.inverse_transform(
    test_predictions_encoded
)

# Create submission
submission_xgb = sample_submission.copy()
submission_xgb["health_condition"] = test_predictions

# Save submission file
submission_xgb.to_csv(
    "submission_xgboost.csv",
    index=False
)

print(submission_xgb.shape)
print(submission_xgb["health_condition"].value_counts())
print(submission_xgb.isnull().sum())

submission_xgb.head()

(295753, 2)
health_condition
at-risk      259392
unhealthy     21165
fit           15196
Name: count, dtype: int64
id                  0
health_condition    0
dtype: int64


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [ ]:
# I did not get a good performance from XGboost. it might be because I have too many categorical data and it did not maybe dealing well with them.

## Conclusion

XGBoost performed significantly worse than both CatBoost and LightGBM on this dataset.

Possible reasons:

- This dataset contains several categorical features. Unlike CatBoost, XGBoost does not natively handle categorical variables, so we had to use one-hot encoding. This can increase the feature dimensionality and may reduce model performance.

- One-hot encoding may have caused the model to lose useful relationships between categorical values that CatBoost can naturally exploit.

- The default hyperparameters used for XGBoost were not heavily tuned. Better tuning (e.g., max_depth, min_child_weight, gamma, learning_rate, subsample, colsample_bytree) could improve performance.

- The feature engineering used here was minimal. XGBoost often benefits from well-engineered numerical and interaction features.

- LightGBM and CatBoost appear to be a better fit for this particular dataset.

Overall, XGBoost achieved an out-of-fold balanced accuracy of approximately **0.881**, which is substantially lower than the **~0.949** achieved by both CatBoost and LightGBM. Therefore, the next stage of experimentation will focus on feature engineering and hyperparameter tuning using the stronger models rather than further optimizing XGBoost.